In [30]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import itertools
import random

In [31]:
BASE_DIR = "/users/6/mehta423/daycent/data/SAS_KGML_090925"
INPUT_DIR = os.path.join(BASE_DIR, "InputData")
OUTPUT_DIR = os.path.join(BASE_DIR, "OutputData_Realistic_8")
PROCESSED_DIR = "/users/6/mehta423/daycent/data/experiment5"
EXPT2_PROCESSED_DIR = "/users/6/mehta423/daycent/data/experiment2"

WEATHER_DIR = os.path.join(INPUT_DIR, "WeatherData")
INITC_FN = os.path.join(INPUT_DIR, "initial_site_conditions.xlsx")

MONTHLY_FN = os.path.join(OUTPUT_DIR, "SAS_scenario_1_monthly.csv")
HARVEST_FN = os.path.join(OUTPUT_DIR, "SAS_scenario_1_harvest.csv")

SCENARIOS_FN = os.path.join(INPUT_DIR, "schedule_scenarios_all_Realistic_8.csv")

In [32]:
all_points = []

for points in os.listdir(WEATHER_DIR):
    df = pd.read_csv(os.path.join(WEATHER_DIR, points))
    df['point_id'] = points.split(".csv")[0]
    all_points.append(df)

weather_df = pd.concat(all_points, ignore_index=True)
weather_df

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,5.270,-4.830,0.0000,1773513
1,2000,2,12.730,-1.850,0.0000,1773513
2,2000,3,15.020,2.430,0.0520,1773513
3,2000,4,10.480,0.620,1.0260,1773513
4,2000,5,1.870,-5.290,0.0410,1773513
...,...,...,...,...,...,...
1853791,2024,362,12.818,-2.313,0.8766,710977
1853792,2024,363,0.784,-5.354,0.0000,710977
1853793,2024,364,4.451,-5.468,0.0062,710977
1853794,2024,365,3.933,-3.492,0.1565,710977


In [33]:
# pids = weather_df['point_id'].unique()
# random.shuffle(pids)
# pids

# train_pids = pids[:100]
# test_pids = pids[100:]

# # save both these as numpy
# np.save(os.path.join(PROCESSED_DIR, "train_pids.npy"), train_pids)
# np.save(os.path.join(PROCESSED_DIR, "test_pids.npy"), test_pids)

train_pids = np.load(os.path.join(EXPT2_PROCESSED_DIR, "train_pids.npy") ,allow_pickle=True)
test_pids = np.load(os.path.join(EXPT2_PROCESSED_DIR, "test_pids.npy") ,allow_pickle=True)

In [34]:
train_weather = weather_df[weather_df['point_id'].isin(train_pids)]

#normalise tmax tmin and precip using standard scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_weather[['Tmax', 'Tmin', 'Precip']] = scaler.fit_transform(train_weather[['Tmax', 'Tmin', 'Precip']])
train_weather

,Year,doy,Tmax,Tmin,Precip,point_id
18264,2000,1,-0.875494,-1.026002,-0.396048,1773749
18265,2000,2,-0.242352,-0.697959,-0.379954,1773749
18266,2000,3,-0.088552,-0.304696,-0.336059,1773749
18267,2000,4,-0.397006,-0.393278,1.132942,1773749
18268,2000,5,-1.198473,-0.954942,-0.396048,1773749
...,...,...,...,...,...,...
1853791,2024,362,-0.240814,-0.700198,0.886547,710977
1853792,2024,363,-1.269050,-0.996216,-0.396048,710977
1853793,2024,364,-0.955726,-1.007313,-0.386977,710977
1853794,2024,365,-0.999986,-0.814964,-0.167066,710977


In [35]:
test_weather = weather_df[weather_df['point_id'].isin(test_pids)]
test_weather[['Tmax', 'Tmin', 'Precip']] = scaler.transform(test_weather[['Tmax', 'Tmin', 'Precip']])
test_weather

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,-0.885747,-0.945208,-0.396048,1773513
1,2000,2,-0.248333,-0.655128,-0.396048,1773513
2,2000,3,-0.052665,-0.238504,-0.319965,1773513
3,2000,4,-0.440582,-0.414693,1.105142,1773513
4,2000,5,-1.176258,-0.989986,-0.336059,1773513
...,...,...,...,...,...,...
1835527,2024,362,-0.198091,-0.692800,2.008197,703163
1835528,2024,363,-1.196679,-1.017339,-0.396048,703163
1835529,2024,364,-0.889507,-0.957376,-0.368102,703163
1835530,2024,365,-1.015452,-0.768532,0.101422,703163


In [36]:
weather_df = pd.concat([train_weather, test_weather], ignore_index=True)
weather_df

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,-0.875494,-1.026002,-0.396048,1773749
1,2000,2,-0.242352,-0.697959,-0.379954,1773749
2,2000,3,-0.088552,-0.304696,-0.336059,1773749
3,2000,4,-0.397006,-0.393278,1.132942,1773749
4,2000,5,-1.198473,-0.954942,-0.396048,1773749
...,...,...,...,...,...,...
1853791,2024,362,-0.198091,-0.692800,2.008197,703163
1853792,2024,363,-1.196679,-1.017339,-0.396048,703163
1853793,2024,364,-0.889507,-0.957376,-0.368102,703163
1853794,2024,365,-1.015452,-0.768532,0.101422,703163


In [47]:
# vocabulary of management events
MANAGEMENT_CLASSES = [
    'conventional_till_molboadplow', 'corn_planting', 'cycle_end', 'harvest_grain', 
    'herbicide', 'nitrogen_fertilization_0gNm2', 'nitrogen_fertilization_1.5gNm2', 
    'nitrogen_fertilization_10.312gNm2', 'nitrogen_fertilization_17.74gNm2', 'notill_rodweederrow', 
    'reduced_till_tandemdisk', 'ryegrass_planting', 'soybean_planting', 'winterwheat_planting']

scenarios_df = pd.read_csv(SCENARIOS_FN)
scenarios_df = scenarios_df.rename({'simyear': 'Year'},  axis=1)

scenarios_df = scenarios_df.pivot_table(
    index=['scenario', 'Year', 'doy'], # Use all identifying columns for the index
    columns='management',
    aggfunc='size',
    fill_value=0
).reset_index()

# Flatten MultiIndex columns if pivot_table created one
if isinstance(scenarios_df.columns, pd.MultiIndex):
    scenarios_df.columns = [c if isinstance(c, str) else c[1] for c in scenarios_df.columns]


# Identify missing management columns
missing = [c for c in MANAGEMENT_CLASSES if c not in scenarios_df.columns]

# Add them with zeros
for col in missing:
    scenarios_df[col] = 0

# Reorder columns so they follow MANAGEMENT_CLASSES after the identifying ones
scenarios_df = scenarios_df[['scenario', 'Year', 'doy'] + MANAGEMENT_CLASSES]
scenarios_df

management,scenario,Year,doy,conventional_till_molboadplow,corn_planting,cycle_end,harvest_grain,herbicide,nitrogen_fertilization_0gNm2,nitrogen_fertilization_1.5gNm2,nitrogen_fertilization_10.312gNm2,nitrogen_fertilization_17.74gNm2,notill_rodweederrow,reduced_till_tandemdisk,ryegrass_planting,soybean_planting,winterwheat_planting
0,1_CS_RT,2000,125,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,1_CS_RT,2000,129,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,1_CS_RT,2000,130,0,1,0,0,0,0,0,0,1,0,0,0,0,0
3,1_CS_RT,2000,305,0,0,1,1,0,0,0,0,0,0,0,0,0,0
4,1_CS_RT,2000,310,0,0,0,0,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
950,8_CS_R2_WW_RT,2023,136,0,0,0,0,1,0,0,0,0,0,0,0,0,0
951,8_CS_R2_WW_RT,2023,137,0,0,0,0,0,0,1,0,0,0,0,0,1,0
952,8_CS_R2_WW_RT,2023,289,0,0,1,1,0,0,0,0,0,0,0,0,0,0
953,8_CS_R2_WW_RT,2023,299,0,0,0,0,0,0,0,1,0,0,0,0,0,1


In [48]:
scenarios = scenarios_df['scenario'].unique()
scenarios

array(['1_CS_RT', '2_CS_CT', '3_CS_R2_RT', '4_CS_R3_RT', '5_CSWW_RT',
       '6_CSWW_CT', '7_CS_R3_WW_RT', '8_CS_R2_WW_RT'], dtype=object)

In [49]:
def load_single_scenario_output(scenario_id: str):
    """Load output data for a single scenario"""
    month_to_doy = {1:30, 2:58, 3:89, 4:119, 5:150, 6:180, 7:211, 8:242, 9:272, 10:303, 11:333, 12:364}
    
    monthly_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"SAS_current_{scenario_id}__monthly.csv"))
    monthly_df = monthly_df.rename({'id': 'point_id'}, axis=1)
    monthly_df['doy'] = monthly_df['month'].map(month_to_doy)
    monthly_df['simyear'] = monthly_df['simyear'].apply(lambda x: math.floor(float(x)))

    harvest_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"SAS_current_{scenario_id}__harvest.csv"))
    harvest_df = harvest_df.rename({'id': 'point_id', 'dayofyr': 'doy'}, axis=1)

    output_df = pd.merge(monthly_df, harvest_df, on=['runid', 'point_id', 'simyear', 'doy'], how='outer')
    output_df = output_df.rename({'simyear': 'Year'}, axis=1)
    output_df['point_id'] = output_df['point_id'].astype(str)

    dates = pd.to_datetime(output_df['Year'].astype(str) + '-' + output_df['doy'].astype(str), format='%Y-%j')
    output_df['month'].fillna(dates.dt.month, inplace=True)
    output_df['month'] = output_df['month'].astype(int)

    output_df.sort_values(['point_id', 'Year', 'month', 'doy'], inplace=True)
    output_df.sort_index(inplace=True)

    output_df['scenario_id'] = scenario_id
    return output_df


def load_output_data(scenario_ids: list[str], max_workers: int = None):
    """Load output data for multiple scenarios using multithreading"""
    all_outputs = []
    
    # Use ThreadPoolExecutor for I/O-bound operations
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_scenario = {
            executor.submit(load_single_scenario_output, scenario_id): scenario_id 
            for scenario_id in scenario_ids
        }
        
        # Collect results as they complete
        for future in tqdm(as_completed(future_to_scenario), "Output scenarios loaded", total=len(future_to_scenario)):
            scenario_id = future_to_scenario[future]
            try:
                output_df = future.result()
                all_outputs.append(output_df)
            except Exception as exc:
                print(f'Scenario {scenario_id} generated an exception: {exc}')
    
    return pd.concat(all_outputs, ignore_index=True)


def load_management_data(scenario_ids: list[str]):
    """Load and process management data for given scenario IDs"""

    scenarios_df = pd.read_csv(SCENARIOS_FN)
    scenarios_df = scenarios_df.rename({'simyear': 'Year'},  axis=1)

    scenarios_df = scenarios_df.pivot_table(
        index=['scenario', 'Year', 'doy'], # Use all identifying columns for the index
        columns='management',
        aggfunc='size',
        fill_value=0
    ).reset_index()

    # Flatten MultiIndex columns if pivot_table created one
    if isinstance(scenarios_df.columns, pd.MultiIndex):
        scenarios_df.columns = [c if isinstance(c, str) else c[1] for c in scenarios_df.columns]


    # Identify missing management columns
    missing = [c for c in MANAGEMENT_CLASSES if c not in scenarios_df.columns]

    # Add them with zeros
    for col in missing:
        scenarios_df[col] = 0

    # Reorder columns so they follow MANAGEMENT_CLASSES after the identifying ones
    scenarios_df = scenarios_df[['scenario', 'Year', 'doy'] + MANAGEMENT_CLASSES]
    # filter for only requested scenarios
    scenarios_df = scenarios_df[scenarios_df['scenario'].isin(scenario_ids)]

    return scenarios_df


def load_data(scenario_ids: list[str], weather_df: pd.DataFrame, max_workers: int = None):
    print("Loading management data...")
    management_df = load_management_data(scenario_ids)

    # unique sets
    scenarios = management_df["scenario"].unique()
    years = weather_df["Year"].unique()
    doys = weather_df["doy"].unique()

    grid = pd.DataFrame(itertools.product(scenarios, years, doys),
                        columns=["scenario", "Year", "doy"])

    grid_weather = pd.merge(grid, weather_df, on=["Year","doy"], how="left")

    X_daily = pd.merge(grid_weather, management_df, 
                    on=["scenario","Year","doy"], 
                    how="left")
    X_daily.fillna(0, inplace=True)
    # drop doy > 365
    X_daily = X_daily[X_daily['doy'] <= 365]
    X_daily.sort_values(['scenario', 'point_id', 'Year', 'doy'], inplace=True)
    X_daily.reset_index(drop=True, inplace=True)

    print("Loading output data...")
    Y = load_output_data(scenario_ids, max_workers=max_workers)

    return X_daily, Y


# Example usage:
X_daily, Y = load_data(scenarios, weather_df, max_workers=8)

print(X_daily.head())
print(Y.head())

Loading management data...
Loading output data...


Output scenarios loaded: 100%|██████████| 8/8 [00:00<00:00, 13.83it/s]

  scenario  Year  doy      Tmax      Tmin    Precip point_id  \
0  1_CS_RT  2000    1 -0.885747 -0.945208 -0.396048  1773513   
1  1_CS_RT  2000    2 -0.248333 -0.655128 -0.396048  1773513   
2  1_CS_RT  2000    3 -0.052665 -0.238504 -0.319965  1773513   
3  1_CS_RT  2000    4 -0.440582 -0.414693  1.105142  1773513   
4  1_CS_RT  2000    5 -1.176258 -0.989986 -0.336059  1773513   

   conventional_till_molboadplow  corn_planting  cycle_end  ...  herbicide  \
0                            0.0            0.0        0.0  ...        0.0   
1                            0.0            0.0        0.0  ...        0.0   
2                            0.0            0.0        0.0  ...        0.0   
3                            0.0            0.0        0.0  ...        0.0   
4                            0.0            0.0        0.0  ...        0.0   

   nitrogen_fertilization_0gNm2  nitrogen_fertilization_1.5gNm2  \
0                           0.0                             0.0   
1           

In [50]:
X_daily

,scenario,Year,doy,Tmax,Tmin,Precip,point_id,conventional_till_molboadplow,corn_planting,cycle_end,...,herbicide,nitrogen_fertilization_0gNm2,nitrogen_fertilization_1.5gNm2,nitrogen_fertilization_10.312gNm2,nitrogen_fertilization_17.74gNm2,notill_rodweederrow,reduced_till_tandemdisk,ryegrass_planting,soybean_planting,winterwheat_planting
0,1_CS_RT,2000,1,-0.885747,-0.945208,-0.396048,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1_CS_RT,2000,2,-0.248333,-0.655128,-0.396048,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1_CS_RT,2000,3,-0.052665,-0.238504,-0.319965,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1_CS_RT,2000,4,-0.440582,-0.414693,1.105142,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1_CS_RT,2000,5,-1.176258,-0.989986,-0.336059,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14818995,8_CS_R2_WW_RT,2024,361,-0.759973,-0.516513,-0.327280,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14818996,8_CS_R2_WW_RT,2024,362,-0.240814,-0.700198,0.886547,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14818997,8_CS_R2_WW_RT,2024,363,-1.269050,-0.996216,-0.396048,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14818998,8_CS_R2_WW_RT,2024,364,-0.955726,-1.007313,-0.386977,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [51]:
Y

,runid,point_id,Year,month,somsc,doy,cgrain,scenario_id
0,1,657200,2000,10,NaN,305,381.653,3_CS_R2_RT
1,1,657200,2001,1,5086.97,30,NaN,3_CS_R2_RT
2,1,657200,2001,2,5088.75,58,NaN,3_CS_R2_RT
3,1,657200,2001,3,5090.38,89,NaN,3_CS_R2_RT
4,1,657200,2001,4,5092.12,119,NaN,3_CS_R2_RT
...,...,...,...,...,...,...,...,...
509931,203,1799675,2024,9,6901.78,272,NaN,7_CS_R3_WW_RT
509932,203,1799675,2024,10,6898.02,303,NaN,7_CS_R3_WW_RT
509933,203,1799675,2024,11,6895.50,333,NaN,7_CS_R3_WW_RT
509934,203,1799675,2024,12,6894.03,364,NaN,7_CS_R3_WW_RT


In [52]:
Y['scenario'] = Y['scenario_id'].apply(lambda x: f'scenario_{x}')

import joblib
# read the scaler_Y
scaler_Y = joblib.load(os.path.join(EXPT2_PROCESSED_DIR, "scaler_Y.pkl"))
train_Y = Y[Y['point_id'].isin(train_pids)]
train_Y[['somsc', 'cgrain']] = scaler_Y.transform(train_Y[['somsc', 'cgrain']]) # change this to fit transform when making scaler
test_Y = Y[Y['point_id'].isin(test_pids)]
test_Y[['somsc', 'cgrain']] = scaler_Y.transform(test_Y[['somsc', 'cgrain']])

Y = pd.concat([train_Y, test_Y])
Y.sort_values(['scenario', 'point_id', 'Year', 'doy'], inplace=True)
Y

,runid,point_id,Year,month,somsc,doy,cgrain,scenario_id,scenario
414794,104,1773513,2000,10,NaN,305,1.607161,1_CS_RT,scenario_1_CS_RT
414795,104,1773513,2001,1,-0.830543,30,NaN,1_CS_RT,scenario_1_CS_RT
414796,104,1773513,2001,2,-0.829097,58,NaN,1_CS_RT,scenario_1_CS_RT
414797,104,1773513,2001,3,-0.827767,89,NaN,1_CS_RT,scenario_1_CS_RT
414798,104,1773513,2001,4,-0.826462,119,NaN,1_CS_RT,scenario_1_CS_RT
...,...,...,...,...,...,...,...,...,...
223563,103,710977,2024,9,-0.292306,272,NaN,8_CS_R2_WW_RT,scenario_8_CS_R2_WW_RT
223564,103,710977,2024,10,-0.295390,303,NaN,8_CS_R2_WW_RT,scenario_8_CS_R2_WW_RT
223565,103,710977,2024,11,-0.297276,333,NaN,8_CS_R2_WW_RT,scenario_8_CS_R2_WW_RT
223566,103,710977,2024,12,-0.298357,364,NaN,8_CS_R2_WW_RT,scenario_8_CS_R2_WW_RT


In [53]:
# save scaler_Y
import joblib
joblib.dump(scaler_Y, os.path.join(PROCESSED_DIR, "scaler_Y.pkl"))

['/users/6/mehta423/daycent/data/experiment5/scaler_Y.pkl']

# Preprocessing

## Inputs processing

In [54]:
df = X_daily[X_daily['point_id'].isin(test_pids)]

# Step 2: Select feature columns (include doy, exclude Year & point_id)
feature_cols = [c for c in df.columns if c not in ["scenario", "Year", "point_id"]]

# Step 3: Group by point_id and Year
groups = df.groupby(["scenario", "Year", "point_id"])

# Step 4: Create sequences and store mapping
sequences = []
mapping = []  # to store (scenario, point_id, year) for each sequence

for (sid, pid, year), group in tqdm(groups):
    sequences.append(group[feature_cols].to_numpy())
    mapping.append((sid, pid, year))  # store mapping info

# Convert to arrays
sequences = np.stack(sequences)  # shape: (num_sequences, 365, num_features)
mapping = np.array(mapping)      # shape: (num_sequences, 2)

# Step 5: Save both sequences and mapping
# Step 5: Save everything into one npy file
data_dict = {
    "data": sequences,
    "mapping": mapping,
    "columns": feature_cols
}

np.save(os.path.join(PROCESSED_DIR, "test_X.npy"), data_dict, allow_pickle=True)



100%|██████████| 20600/20600 [00:07<00:00, 2722.55it/s]


In [55]:
temp1 = np.load(os.path.join(PROCESSED_DIR, "test_X.npy"), allow_pickle=True).item()
temp1['data'].shape, temp1['mapping'].shape, len(temp1['columns'])

print(temp1['columns'])

temp2 = np.load(os.path.join(EXPT2_PROCESSED_DIR, "test_X.npy"), allow_pickle=True).item()
print(temp2['columns'])

# find the difference between two lists
set(temp2['columns']) - set(temp1['columns'])

['doy', 'Tmax', 'Tmin', 'Precip', 'conventional_till_molboadplow', 'corn_planting', 'cycle_end', 'harvest_grain', 'herbicide', 'nitrogen_fertilization_0gNm2', 'nitrogen_fertilization_1.5gNm2', 'nitrogen_fertilization_10.312gNm2', 'nitrogen_fertilization_17.74gNm2', 'notill_rodweederrow', 'reduced_till_tandemdisk', 'ryegrass_planting', 'soybean_planting', 'winterwheat_planting']
['doy', 'Tmax', 'Tmin', 'Precip', 'conventional_till_molboadplow', 'corn_planting', 'cycle_end', 'harvest_grain', 'herbicide', 'nitrogen_fertilization_0gNm2', 'nitrogen_fertilization_1.5gNm2', 'nitrogen_fertilization_10.312gNm2', 'nitrogen_fertilization_17.74gNm2', 'notill_rodweederrow', 'reduced_till_tandemdisk', 'ryegrass_planting', 'soybean_planting', 'winterwheat_planting']


set()

In [56]:
Y_temp = Y[Y['point_id'].isin(test_pids)]
# Build dictionary keyed by (point_id, Year)
Y_dict = {}
for (sid, pid, year), group in tqdm(Y_temp.groupby(["scenario_id", "point_id", "Year"])):
    Y_dict[(sid, pid, year)] = group[["month", "doy", "somsc", "cgrain"]].to_numpy()
# Align Y to mapping
somsc_list = []
cgrain_list = []
for sid, year, pid in tqdm(mapping, desc="Aligning Y to mapping"):
    if (str(sid), str(pid), int(year)) in Y_dict:
        data = Y_dict[str(sid), str(pid), int(year)]

        somsc_array = np.full(12, np.nan, dtype=np.float64)
        
        # The 'data' array has columns: 0=month, 1=doy, 2=somsc, 3=cgrain
        
        # 1a. Create a boolean mask to filter rows where 'somsc' (column index 2) is NOT NaN
        valid_somsc_mask = ~np.isnan(data[:, 2])
        
        # 1b. Filter the data to include only rows with valid somsc values
        valid_data = data[valid_somsc_mask]
        
        # 1c. Get the 0-indexed positions for assignment: month (column 0) - 1
        # We must ensure the indices are integers
        indices = valid_data[:, 0].astype(int) - 1
        
        # 1d. Get the corresponding somsc values (column 2)
        values = valid_data[:, 2]
        
        # 1e. Use advanced NumPy indexing for vectorized assignment
        # This is much faster than iterating row by row.
        # Note: If there are multiple somsc values for the same month, 
        # the last value encountered in the 'data' array (due to sorting in Y_dict) will be used.
        if indices.size > 0:
            somsc_array[indices] = values

            # 2. cgrain value: Must be a single number (no NaNs allowed in the source data)
        # Extract all cgrain values for this year (column index 3)
        cgrain_values = data[:, 3]
        
        # Filter out NaN values to find the single valid cgrain number
        valid_cgrain = cgrain_values[~np.isnan(cgrain_values)]
        
        # Append the results
        if valid_cgrain.size > 0:
            # Append the single annual cgrain value (the user guarantees it's unique/present)
            cgrain_list.append(valid_cgrain[0]) 
            
            # Append the 12-element monthly somsc array
            somsc_list.append(somsc_array)
        else:
            # Handle the case where Cgrain is unexpectedly missing (use NaN as a fallback)
            # print(f"Warning: cgrain value is missing for point_id {pid}, Year {year}. Appending NaN.")
            somsc_list.append(somsc_array)
            cgrain_list.append(np.nan)
        

    else:
        print(f"Missing data for point_id {pid}, Year {year}, scenario {sid}. ")
        somsc_list.append(np.full(12, np.nan, dtype=np.float64))
        cgrain_list.append(np.nan)


# Convert lists to final NumPy arrays
final_somsc_array = np.array(somsc_list)
final_cgrain_array = np.array(cgrain_list)

# --- RESULTS ---
print("\n--- Final Results ---")
print("Mapping length:", len(mapping))
print("SOMSC List length:", len(somsc_list))
print("CGRAIN List length:", len(cgrain_list))

print(f"\nFinal SOMSC Array (Shape: {final_somsc_array.shape}):")
print(final_somsc_array)

print(f"\nFinal CGRAIN Array (Shape: {final_cgrain_array.shape}):")
print(final_cgrain_array)

data_dict = {
    "somsc": final_somsc_array,
    "cgrain": final_cgrain_array,
}

np.save(os.path.join(PROCESSED_DIR, "test_Y.npy"), data_dict, allow_pickle=True)

Aligning Y to mapping: 100%|██████████| 20600/20600 [00:00<00:00, 71481.95it/s]



--- Final Results ---
Mapping length: 20600
SOMSC List length: 20600
CGRAIN List length: 20600

Final SOMSC Array (Shape: (20600, 12)):
[[        nan         nan         nan ...         nan         nan
          nan]
 [        nan         nan         nan ...         nan         nan
          nan]
 [        nan         nan         nan ...         nan         nan
          nan]
 ...
 [ 1.05510953  1.05509291  1.05494331 ...  1.04748823  1.0452193
   1.04390614]
 [ 0.24839037  0.24845686  0.24838206 ...  0.24413508  0.24239805
   0.24135085]
 [-1.61737365 -1.61732379 -1.61738196 ... -1.62034072 -1.62209437
  -1.62307508]]

Final CGRAIN Array (Shape: (20600,)):
[ 1.60716069  1.57879135  1.68904317 ...  0.01301204  0.02119488
 -0.30897609]


In [57]:
mapping

array([['1_CS_RT', '2000', '1773513'],
       ['1_CS_RT', '2000', '1773576'],
       ['1_CS_RT', '2000', '1773883'],
       ...,
       ['8_CS_R2_WW_RT', '2024', '701081'],
       ['8_CS_R2_WW_RT', '2024', '701768'],
       ['8_CS_R2_WW_RT', '2024', '703163']],
      shape=(20600, 3), dtype='<U21')

In [58]:
Y_dict

{('1_CS_RT',
  '1773513',
  np.int64(2000)): array([[ 10.        , 305.        ,          nan,   1.60716069]]),
 ('1_CS_RT',
  '1773513',
  np.int64(2001)): array([[  1.        ,  30.        ,  -0.83054301,          nan],
        [  2.        ,  58.        ,  -0.82909687,          nan],
        [  3.        ,  89.        ,  -0.82776709,          nan],
        [  4.        , 119.        ,  -0.82646225,          nan],
        [  5.        , 150.        ,  -0.82437615,          nan],
        [  6.        , 180.        ,  -0.81741143,          nan],
        [  7.        , 211.        ,  -0.8214257 ,          nan],
        [  8.        , 242.        ,  -0.82260588,          nan],
        [  9.        , 272.        ,  -0.82397722,          nan],
        [ 10.        , 289.        ,          nan,  -0.91018728],
        [ 10.        , 303.        ,  -0.82576411,          nan],
        [ 11.        , 333.        ,  -0.82572256,          nan],
        [ 12.        , 364.        ,  -0.82502442,  